# Intro to AI Engineering — Semester Project Template (Colab)

**What this notebook is:** a semester-long starter that evolves week-by-week into your final project (PM1→PM4).

**What works out of the box:** a tiny toy knowledge base, a working Gradio app, simple retrieval (keyword), simple tool routing, basic guardrails, logging, evaluation, and a tiny A/B experiment harness.

✅ You can run this notebook **without an API key** by leaving `use_mock_llm=True` in the config.

## How to run
1. Run **Setup**
2. Run **All code cells**
3. Run the **Launch Gradio App** cell

## What to customize
- Replace the **Toy Knowledge Base** with your project docs/data
- Replace **keyword retrieval** with embeddings + vector search
- Replace **mock LLM** with your course-approved LLM API
- Expand **guardrails**, **trust signals**, and **evaluation**

---
## Known limitations (starter)
- Keyword retrieval is weak (no embeddings).
- Mock LLM is not “intelligent”; it just formats responses.
- Guardrails are basic and incomplete.


## 1) Setup

In [19]:
# Install lightweight dependencies.
# Keep installs minimal for student reliability.
!pip -q install gradio pandas numpy pypdf openai

import os, re, json, time, math, hashlib
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import numpy as np

# Create folders for logs/results (safe if they already exist)
os.makedirs("runs", exist_ok=True)
os.makedirs("data", exist_ok=True)


In [20]:
# Standard setup cell
# Clones your specific GitHub repository
!git clone --depth 1 -q https://github.com/HrishiKabra/AI_Engineering_Project.git /content/project_repo

!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git /content/main_course_repo
import sys; sys.path.append('/content/main_course_repo')
from course_utils import lab5_setup, get_text_embedding
lab5_setup()


fatal: destination path '/content/project_repo' already exists and is not an empty directory.
fatal: destination path '/content/main_course_repo' already exists and is not an empty directory.
🔧 Setting up your environment...
  → Installing core packages...
  → Setting random seed for reproducible results...
  → Checking API key...
✅ OPENAI_API_KEY already set.
  → Adding course files to path...
✅ Setup complete!
✅ lab5_setup complete — ready.


In [21]:
from pathlib import Path
from pypdf import PdfReader

def pdf_to_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        txt = page.extract_text() or ""
        parts.append(txt)
    return "\n".join(parts)

def load_pdfs(folder: Path):
    docs = []
    for p in sorted(folder.rglob("*.pdf")):
        text = pdf_to_text(p).strip()
        if len(text) > 500:  # avoid tiny/empty extracts
            docs.append({"source": str(p), "text": text})
    return docs

## 2) Config
All experimental knobs live here so you can do controlled comparisons (PM3).

In [22]:
@dataclass
class Config:
    # LLM
    model: str = "mock-model"
    temperature: float = 0.2
    max_tokens: int = 400
    use_mock_llm: bool = False  # ✅ set False when you wire a real API

    # RAG
    use_rag: bool = True
    chunk_size: int = 300
    chunk_overlap: int = 50
    top_k: int = 4

    # Tools / Router
    use_router: bool = True

    # Safety & trust
    guardrails_on: bool = True
    require_citations: bool = True
    low_confidence_threshold: float = 0.25  # heuristic for warnings

    # Logging / Ops
    log_path: str = "runs/interactions.jsonl"

cfg = Config()
cfg


Config(model='mock-model', temperature=0.2, max_tokens=400, use_mock_llm=True, use_rag=True, chunk_size=300, chunk_overlap=50, top_k=4, use_router=True, guardrails_on=True, require_citations=True, low_confidence_threshold=0.25, log_path='runs/interactions.jsonl')

## 3) Data (Knowledge Base)
Start with a toy dataset so the pipeline works immediately. Replace with your project data later.

**Goal:** produce a list of documents with `id`, `title`, and `text` (and optional metadata).

In [23]:
from pathlib import Path
# Point to the cloned repository folder
BASE = Path("/content/project_repo/docs")
REGS = BASE / "regulations"
DECS = BASE / "decision_docs"

reg_docs = load_pdfs(REGS)
dec_docs = load_pdfs(DECS)
docs = reg_docs + dec_docs

print("regulations:", len(reg_docs))
print("decisions:", len(dec_docs))
print("total:", len(docs))
if docs:
    print("example source:", docs[0]["source"])
    print(docs[0]["text"][:800])
else:
    print("No PDFs found. Make sure the github repo URL is correct and contains the docs folder.")


regulations: 2
decisions: 17
total: 19
example source: /content/project_repo/docs/regulations/fia_2025_f1_guidelines_penalty_points_overview.pdf
2025 FORMULA 1 PENALTY AND POINT GUIDELINES  
VERSION - 14 MAY 2025 
 
    
1 
Penalty 
Guidelines 
1 
 
Guidelines for Penalties and Points – 2025 [14 MAY] [CLEAN] 
 
 
 
 
Offence SR Article Free Practice PP Qualifying PP Race PP 
Sanctions 
Receiving five reprimands, four of which were imposed for a driving 
infringement 
18.2 Ten grid places 0 Ten grid places 0 Ten grid places (mandatory) 0 
Personnel Curfew and Covers On Breaches 
Breach of Personnel Curfew 23.6 & 23.5     Pit lane start for both 
Competitor cars (mandatory) 
0 
Breach of “Covers On” Time 23.6 & 38.2a) i)     Pit lane start for both 
Competitor cars (mandatory) 
 
Receiving Mechanical Assistance 
Re-joining by use of mechanical assistance 26.4     Disqualification 0 
Use of Tyres 
Use of tyres without appropriate iden


In [24]:
KB = []

def add_folder_to_kb(folder, prefix):
    for pdf in folder.glob("*.pdf"):
        text = pdf_to_text(pdf)

        if len(text) < 200:
            continue

        KB.append({
            "id": f"{prefix}_{pdf.stem}",
            "title": pdf.stem,
            "text": text,
            "source": str(pdf)
        })

add_folder_to_kb(REGS, "regulation")
add_folder_to_kb(DECS, "decision")

print("documents loaded:", len(KB))
print(KB[0]["title"])
print(KB[0]["text"][:500])

documents loaded: 20
fia_2025_f1_guidelines_penalty_points_overview
2025 FORMULA 1 PENALTY AND POINT GUIDELINES  
VERSION - 14 MAY 2025 
 
    
1 
Penalty 
Guidelines 
1 
 
Guidelines for Penalties and Points – 2025 [14 MAY] [CLEAN] 
 
 
 
 
Offence SR Article Free Practice PP Qualifying PP Race PP 
Sanctions 
Receiving five reprimands, four of which were imposed for a driving 
infringement 
18.2 Ten grid places 0 Ten grid places 0 Ten grid places (mandatory) 0 
Personnel Curfew and Covers On Breaches 
Breach of Personnel Curfew 23.6 & 23.5     Pit lane start fo


### 3.1 Chunking
For the toy KB we keep docs small. For real docs, chunking matters (PM3 experiments often vary chunk size/overlap).

In [25]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    # Simple character-based chunker (good enough to start)
    if chunk_size <= 0:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

def build_chunks(kb: List[Dict[str, Any]], cfg: Config) -> List[Dict[str, Any]]:
    chunks = []
    for doc in kb:
        for i, chunk in enumerate(chunk_text(doc["text"], cfg.chunk_size, cfg.chunk_overlap)):
            chunks.append({
                "chunk_id": f'{doc["id"]}::chunk{i}',
                "doc_id": doc["id"],
                "title": doc["title"],
                "text": chunk,
                "source": doc.get("source", ""),
            })
    return chunks

CHUNKS = build_chunks(KB, cfg)
len(CHUNKS), CHUNKS[0]["chunk_id"]


(1314, 'regulation_fia_2025_f1_guidelines_penalty_points_overview::chunk0')

## 4) Core components
These are written as simple functions so you can swap implementations without rewriting everything.

### 4.1 Logging

In [26]:
def now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def append_jsonl(path: str, record: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def stable_hash(obj: Any) -> str:
    raw = json.dumps(obj, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:12]


### 4.2 LLM wrapper
Start in mock mode. When ready, replace `call_llm_real_api()` with your course-approved API/proxy.

The wrapper returns both text and metadata for logging/ops.

In [27]:
from openai import OpenAI
import os

# --- TODO: Add your OpenAI API Key below or use Colab Secrets ---
# os.environ['OPENAI_API_KEY'] = 'sk-...'
client = OpenAI() # It will automatically pick up the OPENAI_API_KEY from environment

def call_llm_real_api(prompt: str, cfg: Config) -> Dict[str, Any]:
    """
    Calls the real OpenAI API.
    """
    # Default to gpt-4o-mini if 'mock-model' is still in config
    model_name = cfg.model if cfg.model != 'mock-model' else 'gpt-4o-mini'
    
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful AI assistant tasked with answering questions about Formula 1 regulations and decisions."}, 
                {"role": "user", "content": prompt}
            ],
            temperature=cfg.temperature,
            max_tokens=cfg.max_tokens,
        )
        text = response.choices[0].message.content
        
        # Optional usage parsing
        usage = {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    except Exception as e:
        text = f"Error calling OpenAI API: {e}"
        usage = {"prompt_tokens": None, "completion_tokens": None, "total_tokens": None}
        
    return {
        "text": text,
        "usage": usage,
        "model": model_name,
        "latency_ms": 0.0,
        "cost_usd": 0.0
    }

def call_llm_mock(prompt: str, cfg: Config) -> Dict[str, Any]:
    response = (
        "MOCK ANSWER:\n"
        f"- I received your question/prompt: {prompt[:180]!r}\n"
        "- In a real project, this would be answered using the selected tool(s) and/or RAG.\n"
        "- Now providing a concise response placeholder."
    )
    return {
        "text": response,
        "usage": {"prompt_tokens": None, "completion_tokens": None, "total_tokens": None},
        "model": cfg.model,
        "latency_ms": 0.5,
        "cost_usd": 0.0
    }

import time
def call_llm(prompt: str, cfg: Config) -> Dict[str, Any]:
    t0 = time.time()
    if getattr(cfg, 'use_mock_llm', False):
        out = call_llm_mock(prompt, cfg)
    else:
        out = call_llm_real_api(prompt, cfg)
    
    out['latency_ms'] = round((time.time() - t0) * 1000, 2)
    return out


### 4.3 Retrieval (starter: keyword scoring)
Replace later with embeddings + vector search. Keep the function signature the same.

In [28]:
def keyword_score(query: str, text: str) -> float:
    q = re.findall(r"[a-zA-Z0-9']+", query.lower())
    t = re.findall(r"[a-zA-Z0-9']+", text.lower())
    if not q or not t:
        return 0.0
    qset = set(q)
    hits = sum(1 for w in t if w in qset)
    return hits / max(1, len(t))

def retrieve(query: str, chunks: List[Dict[str, Any]], cfg: Config) -> Tuple[List[Dict[str, Any]], float]:
    scored = []
    for c in chunks:
        s = keyword_score(query, c["text"] + " " + c["title"])
        scored.append((s, c))
    scored.sort(key=lambda x: x[0], reverse=True)

    top = [c for s, c in scored[:cfg.top_k] if s > 0]
    conf = float(min(1.0, scored[0][0])) if scored else 0.0
    return top, conf


### 4.4 Tools (calculator + RAG search)
Tools are thin wrappers so the router/pipeline can call them uniformly.

In [29]:
def safe_calculator(expr: str) -> str:
    if len(expr) > 80:
        return "Expression too long."
    if re.search(r"[A-Za-z_]|\.|\[|\]|\{|\}|;|:", expr):
        return "Unsupported expression."
    try:
        val = eval(expr, {"__builtins__": {}}, {})
        if isinstance(val, (int, float)) and (not math.isfinite(val) or abs(val) > 1e12):
            return "Result out of range."
        return str(val)
    except Exception:
        return "Could not evaluate expression."

def tool_rag_search(query: str, cfg: Config) -> Dict[str, Any]:
    hits, conf = retrieve(query, CHUNKS, cfg)
    return {"hits": hits, "confidence": conf}

def tool_calculate(query: str, cfg: Config) -> Dict[str, Any]:
    return {"result": safe_calculator(query)}


### 4.5 Guardrails + trust signals (starter)
Keep it simple: basic refusals + injection stripping + confidence warnings.

In [30]:
UNSAFE_TERMS = [
    "how to build a bomb",
    "make a weapon",
]

def guardrail_check_input(user_input: str) -> Optional[str]:
    text = user_input.lower().strip()
    if len(text) > 2000:
        return "I can't process inputs that long in this demo. Please shorten your question."
    for term in UNSAFE_TERMS:
        if term in text:
            return "I can't help with that request."
    return None

def strip_prompt_injection(retrieved_text: str) -> str:
    lines = retrieved_text.splitlines()
    cleaned = []
    for ln in lines:
        if re.search(r"(ignore previous|system prompt|you are chatgpt|developer message)", ln.lower()):
            continue
        cleaned.append(ln)
    return "\n".join(cleaned)

def guardrail_check_output(answer: str) -> Optional[str]:
    # TODO: add domain-specific output rules, e.g. no medical/legal advice.
    return None


### 4.6 Prompt templates
Keep prompts in functions so you can run PM3 experiments on prompt styles.

In [31]:
def build_answer_prompt(user_input: str, retrieved: List[Dict[str, Any]]) -> str:
    sources_block = ""
    if retrieved:
        formatted = []
        for i, c in enumerate(retrieved, start=1):
            snippet = strip_prompt_injection(c["text"])
            formatted.append(f"[{i}] {c['title']}: {snippet}")
        sources_block = "\n\nSOURCES:\n" + "\n\n".join(formatted)

    return (
        "You are a helpful assistant.\n"
        "If you use sources, cite them like [1], [2].\n"
        "If you are unsure, say so and ask a follow-up question.\n\n"
        f"USER QUESTION:\n{user_input}\n"
        f"{sources_block}\n\n"
        "ANSWER:"
    )


### 4.7 Pipeline orchestrator
The **single** entry point: `run_pipeline(user_input, cfg)`.

This makes Gradio + evaluation + experiments easy.

In [32]:
@dataclass
class PipelineResult:
    answer: str
    sources: List[Dict[str, Any]]
    tool_trace: List[str]
    warnings: List[str]
    metadata: Dict[str, Any]

def route(user_input: str, cfg: Config) -> str:
    if not cfg.use_router:
        return "direct"
    if re.search(r"\d", user_input) and any(op in user_input for op in ["+", "-", "*", "/", "**"]):
        return "calculator"
    return "rag" if cfg.use_rag else "direct"

def run_pipeline(user_input: str, cfg: Config) -> PipelineResult:
    tool_trace = []
    warnings = []

    # Input guardrails
    if cfg.guardrails_on:
        refusal = guardrail_check_input(user_input)
        if refusal:
            return PipelineResult(
                answer=refusal,
                sources=[],
                tool_trace=["guardrail->refuse"],
                warnings=[],
                metadata={"ts": now_ts(), "cfg_hash": stable_hash(asdict(cfg))},
            )

    r = route(user_input, cfg)
    tool_trace.append(f"router->{r}")

    retrieved = []
    retrieval_conf = 0.0

    if r == "calculator":
        out = tool_calculate(user_input, cfg)
        answer = f"The result is: {out['result']}"
        tool_trace.append("tool->calculator")

    elif r == "rag":
        out = tool_rag_search(user_input, cfg)
        retrieved = out["hits"]
        retrieval_conf = float(out["confidence"])
        tool_trace.append(f"tool->rag_search (hits={len(retrieved)})")

        if cfg.require_citations and not retrieved:
            warnings.append("No sources retrieved. Answer may be unreliable.")
        if retrieval_conf < cfg.low_confidence_threshold:
            warnings.append("Low retrieval confidence — consider verifying sources or rephrasing.")

        prompt = build_answer_prompt(user_input, retrieved)
        llm_out = call_llm(prompt, cfg)
        answer = llm_out["text"]

    else:
        prompt = build_answer_prompt(user_input, [])
        llm_out = call_llm(prompt, cfg)
        answer = llm_out["text"]

    # Output guardrails
    if cfg.guardrails_on:
        out_refusal = guardrail_check_output(answer)
        if out_refusal:
            tool_trace.append("guardrail->output_block")
            answer = out_refusal

    result = PipelineResult(
        answer=answer,
        sources=[{"title": c["title"], "source": c.get("source",""), "snippet": c["text"][:280]} for c in retrieved],
        tool_trace=tool_trace,
        warnings=warnings,
        metadata={
            "ts": now_ts(),
            "route": r,
            "retrieval_conf": retrieval_conf,
            "cfg_hash": stable_hash(asdict(cfg)),
        }
    )

    append_jsonl(cfg.log_path, {
        "ts": result.metadata["ts"],
        "user_input": user_input,
        "answer": result.answer,
        "sources": result.sources,
        "tool_trace": result.tool_trace,
        "warnings": result.warnings,
        "metadata": result.metadata,
        "cfg": asdict(cfg),
    })

    return result


## 5) Gradio App
This UI is intentionally simple but includes trust signals (sources, trace, warnings).

In [33]:
import gradio as gr

def gradio_respond(message: str, history: List[Tuple[str, str]]):
    res = run_pipeline(message, cfg)
    history = history + [(message, res.answer)]

    if res.sources:
        sources_md = "\n".join([f"- **{s['title']}** — {s['source']}\n  - {s['snippet']}" for s in res.sources])
    else:
        sources_md = "_No sources retrieved._"

    trace_md = "\n".join([f"- {t}" for t in res.tool_trace]) or "_(none)_"
    warn_md = "\n".join([f"- ⚠️ {w}" for w in res.warnings]) or "_(none)_"

    return history, sources_md, trace_md, warn_md

with gr.Blocks() as demo:
    gr.Markdown("# Project Demo UI (Starter)")
    gr.Markdown("Tip: Ask about **library hours** or **dining hours** to see retrieval in action.")
    chatbot = gr.Chatbot(label="Chat", height=280)
    msg = gr.Textbox(label="Your message", placeholder="Type a question and press Enter...")
    sources = gr.Markdown(label="Sources")
    trace = gr.Markdown(label="Tool trace")
    warnings = gr.Markdown(label="Warnings")

    def clear():
        return [], "", "", ""

    msg.submit(gradio_respond, inputs=[msg, chatbot], outputs=[chatbot, sources, trace, warnings])
    gr.Button("Clear").click(clear, outputs=[chatbot, sources, trace, warnings])

demo


/tmp/ipykernel_939/4159806487.py:20: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat", height=280)
/tmp/ipykernel_939/4159806487.py:20: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat", height=280)


Gradio Blocks instance: 2 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7830adf6fc20>
 |-<gradio.components.chatbot.Chatbot object at 0x7830b5ff4ad0>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x7830b5ff4ad0>
 |-<gradio.components.markdown.Markdown object at 0x7830b44ff710>
 |-<gradio.components.markdown.Markdown object at 0x7830e5ef4d10>
 |-<gradio.components.markdown.Markdown object at 0x7830b4ea6ba0>
fn_index=1
 inputs:
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x7830b5ff4ad0>
 |-<gradio.components.markdown.Markdown object at 0x7830b44ff710>
 |-<gradio.components.markdown.Markdown object at 0x7830e5ef4d10>
 |-<gradio.components.markdown.Markdown object at 0x7830b4ea6ba0>

In [34]:
# --- Launch Gradio App ---
# In Colab, you can set share=True to get a public link (if allowed by your course policy).
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26c3dc3ec36a7f47fc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6) Evaluation harness (starter)
Create a tiny test set early. Expand it through the semester.

Starter metrics:
- citation marker rate (contains [1])
- sources retrieved rate
- calculator route rate

In [ ]:
TOY_TESTS = [
    {"id": "t1", "question": "What time does the library close on Monday?", "expected": "should mention 10pm and cite sources"},
    {"id": "t2", "question": "What are weekend dining hall hours?", "expected": "should mention 9am–7pm and cite sources"},
    {"id": "t3", "question": "2+2*5", "expected": "should use calculator tool"},
]

def contains_citation(text: str) -> bool:
    return bool(re.search(r"\[\d+\]", text))

def run_eval(cfg_eval: Config, tests: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for ex in tests:
        res = run_pipeline(ex["question"], cfg_eval)
        rows.append({
            "id": ex["id"],
            "question": ex["question"],
            "route": res.metadata.get("route"),
            "has_sources": len(res.sources) > 0,
            "has_citation_marker": contains_citation(res.answer),
            "warnings_count": len(res.warnings),
            "answer": res.answer[:400],
            "human_rating_1to5": np.nan,  # TODO: fill manually or via a rubric tool
        })
    return pd.DataFrame(rows)

def summarize_eval(df: pd.DataFrame) -> Dict[str, Any]:
    return {
        "n": int(len(df)),
        "has_sources_rate": float(df["has_sources"].mean()),
        "citation_marker_rate": float(df["has_citation_marker"].mean()),
        "avg_warnings": float(df["warnings_count"].mean()),
        "calculator_route_rate": float((df["route"] == "calculator").mean()),
    }

df_eval = run_eval(cfg, TOY_TESTS)
display(df_eval)
display(summarize_eval(df_eval))


,id,question,route,has_sources,has_citation_marker,warnings_count,answer,human_rating_1to5
0,t1,What time does the library close on Monday?,rag,True,True,0,MOCK ANSWER:\n- I received your question/promp...,NaN
1,t2,What are weekend dining hall hours?,rag,True,True,1,MOCK ANSWER:\n- I received your question/promp...,NaN
2,t3,2+2*5,calculator,False,False,0,The result is: 12,NaN


{'n': 3,
 'has_sources_rate': 0.6666666666666666,
 'citation_marker_rate': 0.6666666666666666,
 'avg_warnings': 0.3333333333333333,
 'calculator_route_rate': 0.3333333333333333}

## 7) Experiments (PM3 harness starter)
Run controlled A/B comparisons by changing **one** config variable.

Example experiment: `use_rag=True` vs `use_rag=False` on the same test set.

In [ ]:
def run_ab_experiment(cfg_a: Config, cfg_b: Config, tests: List[Dict[str, Any]]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df_a = run_eval(cfg_a, tests)
    df_b = run_eval(cfg_b, tests)

    sum_a = summarize_eval(df_a)
    sum_b = summarize_eval(df_b)

    rows = []
    for k in sorted(sum_a.keys()):
        a, b = sum_a[k], sum_b[k]
        rows.append({"metric": k, "A": a, "B": b, "delta_B_minus_A": (b - a) if isinstance(a, (int, float)) else None})
    return pd.DataFrame(rows), df_a, df_b

cfgA = Config(**asdict(cfg))
cfgB = Config(**asdict(cfg))
cfgA.use_rag = True
cfgB.use_rag = False

summary_df, details_A, details_B = run_ab_experiment(cfgA, cfgB, TOY_TESTS)
summary_df


,metric,A,B,delta_B_minus_A
0,avg_warnings,0.333333,0.000000,-0.333333
1,calculator_route_rate,0.333333,0.333333,0.000000
2,citation_marker_rate,0.666667,0.666667,0.000000
3,has_sources_rate,0.666667,0.000000,-0.666667
4,n,3.000000,3.000000,0.000000


## 8) Project writeup (living doc)

Fill these sections over time. For PM1/PM2/PM3/PM4, you can export this notebook or copy sections into a report.

### PM1 — Proposal (Week 3)
- Team members:
- Title:
- Problem statement & user:
- Why AI / LLM:
- Example interactions (3–5):
- Possible data sources:
- Initial risks/concerns:

### PM2 — MVP notes (Week 6)
- What works right now?
- One obvious failure mode observed:

### PM3 — Experiments (Week 11)
For each experiment:
- Question:
- Hypothesis:
- Design (what changed, what stayed constant):
- Metric:
- Results:
- Conclusion:

### PM4 — Final report outline (Week 15)
1. Introduction
2. System overview (diagram)
3. Components (RAG/agents/guardrails/logging)
4. GenAI Ops experiments
5. Limitations & risks
6. Future work


## 9) Demo mode
Use this section to keep your final presentation stable: set a known-good config and keep 5–10 canned questions ready.

In [ ]:
DEMO_QUESTIONS = [
    "What time does the library close on Thursday?",
    "What are weekend dining hours?",
    "2+2*5",
    "What are the project requirements?",
]

demo_cfg = Config(**asdict(cfg))
demo_cfg.use_mock_llm = True  # flip False when you're ready and have API access
demo_cfg.use_rag = True
demo_cfg.guardrails_on = True

DEMO_QUESTIONS, demo_cfg


(['What time does the library close on Thursday?',
  'What are weekend dining hours?',
  '2+2*5',
  'What are the project requirements?'],
 Config(model='mock-model', temperature=0.2, max_tokens=400, use_mock_llm=True, use_rag=True, chunk_size=300, chunk_overlap=50, top_k=4, use_router=True, guardrails_on=True, require_citations=True, low_confidence_threshold=0.25, log_path='runs/interactions.jsonl'))